In [ ]:
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


APPLY_URL = "https://www.gov.kr/mw/AA040OfferMainFrm.do?capp_biz_cd=13100000026&HighCtgCD=-&FAX_TYPE=B&img=02&selectedSeq=01"

TARGETS = [
    {
        "DONG_SEARCH": "울산광역시 동구 방어동",
        "DONG_SELECT": "울산광역시 동구 방어동(방어동)",
        "PARCELS": [
            (1283, 7),
        ]
    },
    {
        "DONG_SEARCH": "울산광역시 남구 매암동",
        "DONG_SELECT": "울산광역시 남구 야음장생포동(매암동)",
        "PARCELS": [
            (139, 43),
            (472, 50),
        ]
    },
    {
        "DONG_SEARCH": "울산광역시 남구 황성동",
        "DONG_SELECT": "울산광역시 남구 선암동(황성동)",
        "PARCELS": [
            (842, 1),
            (842, 2),
            (843, None),
            (861, None),
            (862, None),
            (863, 1),
            (864, 1),
        ]
    },
]


def click_strong(driver, elem):
    methods = [
        lambda: elem.click(),
        lambda: driver.execute_script("arguments[0].click();", elem),
        lambda: driver.execute_script("""
            const e = arguments[0];
            e.dispatchEvent(new MouseEvent('mousedown', {bubbles:true}));
            e.dispatchEvent(new MouseEvent('mouseup', {bubbles:true}));
            e.dispatchEvent(new MouseEvent('click', {bubbles:true}));
        """, elem),
        lambda: ActionChains(driver).move_to_element(elem).pause(0.2).click().perform(),
    ]
    for method in methods:
        try:
            method()
            return True
        except:
            pass
    return False


def close_survey_popup(driver):
    xpaths = [
        "//button[normalize-space()='다음에 하기']",
        "//*[normalize-space()='다음에 하기']",
        "//button[contains(., '다음에 하기')]",
        "//*[contains(normalize-space(.), '다음에 하기')]",
    ]
    for xp in xpaths:
        try:
            elem = WebDriverWait(driver, 1).until(
                EC.presence_of_element_located((By.XPATH, xp))
            )
            if click_strong(driver, elem):
                return
        except:
            pass
    try:
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    except:
        pass


def select_address_result(driver, wait, dong_select):
    result = wait.until(EC.presence_of_element_located((
        By.XPATH,
        f"//dd[.//div[contains(normalize-space(.), '{dong_select}')]]"
    )))
    if not click_strong(driver, result):
        raise Exception(f"주소 선택 실패: {dong_select}")


def go_apply_page(driver):
    driver.get(APPLY_URL)


def process_one(driver, wait, dong_search, dong_select, bonbun, bubun):
    main_window = driver.current_window_handle
    before_windows = driver.window_handles[:]

    addr_btn = wait.until(EC.presence_of_element_located((By.ID, "btnAddress")))
    if not click_strong(driver, addr_btn):
        raise Exception("주소검색 버튼 클릭 실패")

    wait.until(lambda d: len(d.window_handles) > len(before_windows))
    popup = [w for w in driver.window_handles if w not in before_windows][0]
    driver.switch_to.window(popup)

    dong_input = wait.until(EC.element_to_be_clickable((By.ID, "txtAddr")))
    dong_input.clear()
    dong_input.send_keys(dong_search)
    dong_input.send_keys(Keys.ENTER)

    select_address_result(driver, wait, dong_select)

    wait.until(lambda d: len(d.window_handles) == 1)
    driver.switch_to.window(main_window)

    bonbun_input = wait.until(EC.element_to_be_clickable((
        By.ID,
        "토지임야대장신청서_IN-토지임야대장신청서_신청토지소재지_주소정보_상세주소_번지"
    )))
    bonbun_input.clear()
    bonbun_input.send_keys(str(bonbun))

    bubun_input = wait.until(EC.element_to_be_clickable((
        By.ID,
        "토지임야대장신청서_IN-토지임야대장신청서_신청토지소재지_주소정보_상세주소_호"
    )))
    bubun_input.clear()
    if bubun is not None:
        bubun_input.send_keys(str(bubun))

    apply_btn = wait.until(EC.presence_of_element_located((By.ID, "btn_end")))
    if not click_strong(driver, apply_btn):
        raise Exception("신청하기 버튼 클릭 실패")

    time.sleep(0.8)
    close_survey_popup(driver)
    time.sleep(1.5)


def main():
    options = Options()
    options.add_experimental_option("detach", True)

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 20)

    go_apply_page(driver)

    # 로그인할 시간만 잠깐 줌
    print("30초 안에 정부24 로그인하세요.")
    time.sleep(30)

    total_count = sum(len(t["PARCELS"]) for t in TARGETS)
    done_count = 0
    fail_list = []

    for target in TARGETS:
        dong_search = target["DONG_SEARCH"]
        dong_select = target["DONG_SELECT"]
        parcels = target["PARCELS"]

        print(f"\n=== {dong_search} 시작 / {len(parcels)}건 ===")

        for bonbun, bubun in parcels:
            done_count += 1
            parcel_text = f"{bonbun}" if bubun is None else f"{bonbun}-{bubun}"
            print(f"[{done_count}/{total_count}] {dong_search} {parcel_text}")

            try:
                go_apply_page(driver)
                process_one(driver, wait, dong_search, dong_select, bonbun, bubun)
                print(f"완료: {dong_search} {parcel_text}")
            except Exception as e:
                print(f"실패: {dong_search} {parcel_text} -> {e}")
                fail_list.append((dong_search, dong_select, bonbun, bubun, str(e)))
                continue

    print("\n전체 완료")
    print("실패 건수:", len(fail_list))

    if fail_list:
        print("\n실패 목록")
        for item in fail_list:
            dong_search, dong_select, bonbun, bubun, err = item
            parcel_text = f"{bonbun}" if bubun is None else f"{bonbun}-{bubun}"
            print(f"- {dong_search} {parcel_text} / {err}")


if __name__ == "__main__":
    main()